**Making HTTP request to the Amazon**

In [10]:
!pip install selenium webdriver-manager
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get install -f # Install dependencies if any are missing

--2026-03-29 18:46:41--  https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
Resolving dl.google.com (dl.google.com)... 142.250.31.91, 142.250.31.190, 142.250.31.136, ...
Connecting to dl.google.com (dl.google.com)|142.250.31.91|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 127723048 (122M) [application/x-debian-package]
Saving to: ‘google-chrome-stable_current_amd64.deb’

google-chrome-stabl 100%[===================>] 121.81M   170MB/s    in 0.7s    

2026-03-29 18:46:42 (170 MB/s) - ‘google-chrome-stable_current_amd64.deb’ saved [127723048/127723048]

(Reading database ... 119137 files and directories currently installed.)
Preparing to unpack google-chrome-stable_current_amd64.deb ...
Unpacking google-chrome-stable (146.0.7680.164-1) over (146.0.7680.164-1) ...
Setting up google-chrome-stable (146.0.7680.164-1) ...
Processing triggers for mailcap (3.70+nmu1ubuntu1) ...
Processing triggers for man-db (2.10.2-1) ...
Reading package list

In [12]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time

def get_reviews_selenium(url):
    # Configure Chrome options for headless execution in Colab
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    # Explicitly set the Chrome binary location to google-chrome-stable
    chrome_options.binary_location = '/opt/google/chrome/google-chrome'

    # Use ChromeDriverManager to automatically download and manage chromedriver
    service = webdriver.ChromeService(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    driver.get(url)
    time.sleep(3)

    reviews = []

    review_blocks = driver.find_elements(By.XPATH, '//div[@data-hook="review"]')

    for review in review_blocks:
        try:
            title = review.find_element(By.XPATH, './/a[@data-hook="review-title"]').text
        except:
            title = None

        try:
            rating = review.find_element(By.XPATH, './/i[@data-hook="review-star-rating"]').text.split()[0]
        except:
            rating = None

        try:
            body = review.find_element(By.XPATH, './/span[@data-hook="review-body"]').text
        except:
            body = None

        reviews.append({
            "title": title,
            "rating": float(rating) if rating else None,
            "review_text": body
        })

    driver.quit()
    return reviews


if __name__ == "__main__":
    # Example: Replace with an actual Amazon product ID for a product like 'Echo Dot' or 'Kindle Paperwhite'
    # For demonstration, let's use a placeholder for a common product, e.g., an Echo Dot 5th Gen (B09B8V1FW3)
    url = "https://www.amazon.com/Echo-Dot-5th-Gen/product-reviews/B09B8V1FW3"  # Example Product ID used for demonstration
    data = get_reviews_selenium(url)

    print(f"Scraped {len(data)} reviews")

Scraped 0 reviews


**PreProcessing Pipeline**

In [ ]:
import re
import pandas as pd

def clean_text(text):
    if not text:
        return ""

    # remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    # lowercase
    text = text.lower()

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def remove_duplicates(df):
    return df.drop_duplicates(subset=["review_text"])


def handle_missing_values(df):
    df = df.dropna(subset=["review_text", "rating"])
    return df


def preprocess_data(reviews):
    df = pd.DataFrame(reviews)

    # handle missing values
    df = handle_missing_values(df)

    # clean text
    df["cleaned_review"] = df["review_text"].apply(clean_text)

    # remove duplicates
    df = remove_duplicates(df)

    return df


if __name__ == "__main__":
    # Example usage
    sample_reviews = [
        {"title": "Great!", "rating": 5, "review_text": "<b>Awesome product!!</b>"},
        {"title": "Bad", "rating": 1, "review_text": "Terrible experience..."},
        {"title": "Great!", "rating": 5, "review_text": "<b>Awesome product!!</b>"}
    ]

    df = preprocess_data(sample_reviews)
    print(df.head())

    title  rating               review_text       cleaned_review
0  Great!       5  <b>Awesome product!!</b>      awesome product
1     Bad       1    Terrible experience...  terrible experience
